# CompactLLM — QLoRA fine-tune on Kaggle

Kaggle gives 30 GPU-hours/week (P100 or 2×T4) and rarely disconnects, so a
~2h training run finishes with room to spare. The dataset is in the repo now,
so this is clone + Run all — no file uploads.

**Before you run (right sidebar → Notebook options):**
1. **Accelerator**: GPU T4 x2 (or P100)
2. **Internet**: On
3. **Add-ons → Secrets**: add `HF_TOKEN` (your HF read token). Optional:
   `WANDB_API_KEY`, `GROQ_API_KEY` (eval rationale judge)

Also accept the Gemma license once: https://huggingface.co/google/gemma-3-4b-it

In [ ]:
!nvidia-smi -L

In [ ]:
%cd /kaggle/working
!rm -rf compact-llm && git clone --depth 1 https://github.com/aayush-arya/compact-llm.git
%cd compact-llm
!wc -l data/processed/*.jsonl

In [ ]:
# If unsloth ever conflicts with Kaggle's preinstalled torch, fall back to:
#   !pip install -q -r training/requirements.txt
!pip install -q unsloth
!pip install -q wandb httpx

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

sec = UserSecretsClient()
os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
for opt in ('WANDB_API_KEY', 'GROQ_API_KEY', 'CEREBRAS_API_KEY', 'GEMINI_API_KEY'):
    try:
        os.environ[opt] = sec.get_secret(opt)
    except Exception:
        pass

In [ ]:
# Train -- ~2h on a T4, ~40-60 min on a P100. Writes outputs/adapter/ + outputs/merged/.
#
# To SKIP training and just (re-)evaluate an adapter you already have: upload
# the adapter folder as a Kaggle Dataset named "compactllm-adapter" (Datasets ->
# New Dataset -> drag in adapter_config.json, adapter_model.safetensors, the
# tokenizer files), then "+ Add Input" it to this notebook. This cell detects it.
import pathlib
import shutil

_pre = pathlib.Path("/kaggle/input/compactllm-adapter")
if (_pre / "adapter_config.json").exists():
    pathlib.Path("outputs").mkdir(exist_ok=True)
    shutil.copytree(_pre, "outputs/adapter", dirs_exist_ok=True)
    print(f"Found prebuilt adapter at {_pre} -- skipping training.")
else:
    get_ipython().system("python training/train_unsloth_qlora.py")

In [ ]:
# The 8 GB merged fp16 model is only for a GGUF/Ollama export -- drop it so it
# doesn't bloat the Kaggle output. Comment this out if you need it.
!rm -rf outputs/merged

In [ ]:
# Held-out eval: base zero-shot vs few-shot vs fine-tuned. ~35-40 min on a T4.
!python training/eval_base_vs_finetuned.py --judge auto

In [ ]:
# Copy what you need into /kaggle/working so it's in the notebook's Output tab.
!mkdir -p /kaggle/working/out
!cp -r outputs/adapter /kaggle/working/out/
!cp docs/benchmark_results.json docs/benchmark_table.md /kaggle/working/out/ 2>/dev/null || true
!cd /kaggle/working && zip -r compactllm-outputs.zip out
from IPython.display import FileLink
FileLink('/kaggle/working/compactllm-outputs.zip')

## Get the files off Kaggle

Click the `compactllm-outputs.zip` link above, **or** right sidebar → **Output**
→ download `compactllm-outputs.zip`. (If the link 404s, the session is still
running — wait for the cell to finish.)

### Back on your machine

```bash
unzip compactllm-outputs.zip     # -> out/adapter/, out/benchmark_results.json
```

Then tell Claude — it commits `benchmark_results.json`, pushes the adapter to a
HF model repo, and switches the backend to `MODEL_BACKEND=transformers`.